In [0]:
from pyspark.sql import functions as F
from datetime import datetime
import re

# Configuration for both fact tables
fact_configs = {
    "fact_shift_raw": {
        "key": "shift_id",
        "columns": [
            "shift_id",
            "people_id",
            "county_id",
            "scheduled_start",
            "scheduled_end"
        ],
        "output": "workforce_analytics.silver.fact_shift_snapshot_test"
    },

    "fact_shrinkage_raw": {
        "key": "shrinkage_id",
        "columns": [
            "shrinkage_id",
            "people_id",
            "shrinkage_type_id",
            "start_datetime",
            "end_datetime"
        ],
        "output": "workforce_analytics.silver.fact_shrinkage_snapshot_test"
    }
}

for source_table, config in fact_configs.items():

    print(f"\n========== Processing {source_table} ==========")

    key = config["key"]
    business_columns = config["columns"]
    output_table = config["output"]

    df = spark.table(
        f"workforce_analytics.bronze.{source_table}"
    )

    # Prepare snapshot date and data types
    prepared = (
        df
        .withColumn(
            "snapshot_date",
            F.to_date(
                F.regexp_extract(
                    F.col("_source_file"),
                    r"(\d{2}_\d{2}_\d{4})",
                    1
                ),
                "dd_MM_yyyy"
            )
        )
        .withColumn(key, F.col(key).cast("long"))
        .withColumn("people_id", F.col("people_id").cast("long"))
    )

    if source_table == "fact_shift_raw":

        prepared = (
            prepared
            .withColumn("county_id", F.col("county_id").cast("long"))
            .withColumn(
                "scheduled_start",
                F.to_timestamp("scheduled_start")
            )
            .withColumn(
                "scheduled_end",
                F.to_timestamp("scheduled_end")
            )
        )

    else:

        prepared = (
            prepared
            .withColumn(
                "shrinkage_type_id",
                F.col("shrinkage_type_id").cast("long")
            )
            .withColumn(
                "start_datetime",
                F.to_timestamp("start_datetime")
            )
            .withColumn(
                "end_datetime",
                F.to_timestamp("end_datetime")
            )
        )

    state_columns = business_columns + [
        "_source_file",
        "_ingested_at"
    ]

    # Sort files chronologically
    files = [
        row["_source_file"]
        for row in prepared.select("_source_file").distinct().collect()
    ]

    files = sorted(
        files,
        key=lambda x: datetime.strptime(
            re.search(r"(\d{2}_\d{2}_\d{4})", x).group(1),
            "%d_%m_%Y"
        )
    )

    state = None
    snapshots = []

    # Process initial load and deltas
    for index, file in enumerate(files):

        print(f"Processing {file.split('/')[-1]}")

        current = (
            prepared
            .filter(F.col("_source_file") == file)
            .select(*state_columns)
            .dropDuplicates(business_columns)
        )

        # Stop if an ID has conflicting records within a file
        duplicate_ids = (
            current.groupBy(key)
            .count()
            .filter(F.col("count") > 1)
        )

        if duplicate_ids.limit(1).count() > 0:
            raise ValueError(
                f"Conflicting duplicate IDs found in {file}"
            )

        snapshot_date = (
            prepared
            .filter(F.col("_source_file") == file)
            .select("snapshot_date")
            .first()["snapshot_date"]
        )

        if index == 0:
            state = current
        else:
            delta_ids = current.select(key).distinct()

            unchanged_previous = state.join(
                delta_ids,
                on=key,
                how="left_anti"
            )

            state = unchanged_previous.unionByName(current)

        snapshots.append(
            state.withColumn(
                "snapshot_date",
                F.lit(snapshot_date).cast("date")
            )
        )

        print(
            "Snapshot date:",
            snapshot_date,
            "| Records:",
            state.count()
        )

    # Combine daily snapshots
    final_df = snapshots[0]

    for snapshot in snapshots[1:]:
        final_df = final_df.unionByName(snapshot)

    # Save to new test table
    (
        final_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(output_table)
    )

    print(f"Saved successfully: {output_table}")

In [0]:
from pyspark.sql import functions as F

test_tables = [
    ("fact_shift_snapshot_test", "shift_id"),
    ("fact_shrinkage_snapshot_test", "shrinkage_id")
]

for table, key in test_tables:

    print(f"\n========== {table} ==========")

    df = spark.table(
        f"workforce_analytics.silver.{table}"
    )

    print("Snapshot row counts:")
    display(
        df.groupBy("snapshot_date")
        .count()
        .orderBy("snapshot_date")
    )

    print("Duplicate IDs:")
    display(
        df.groupBy("snapshot_date", key)
        .count()
        .filter(F.col("count") > 1)
    )

In [0]:
from pyspark.sql import functions as F

validation_configs = {
    "fact_shift_snapshot_test": {
        "key": "shift_id",
        "start": "scheduled_start",
        "end": "scheduled_end"
    },
    "fact_shrinkage_snapshot_test": {
        "key": "shrinkage_id",
        "start": "start_datetime",
        "end": "end_datetime"
    }
}

for table, config in validation_configs.items():

    print(f"\n========== {table} ==========")

    df = spark.table(
        f"workforce_analytics.silver.{table}"
    )

    key = config["key"]
    start = config["start"]
    end = config["end"]

    print("Missing IDs:")
    print(
        df.filter(F.col(key).isNull()).count()
    )

    print("Missing timestamps:")
    print(
        df.filter(
            F.col(start).isNull() |
            F.col(end).isNull()
        ).count()
    )

    print("Invalid time ranges:")
    print(
        df.filter(
            F.col(end) <= F.col(start)
        ).count()
    )

### Step 1 — Replace the old Silver fact tables

In [0]:
# Promote the validated snapshot tables to the main Silver tables

promotions = {
    "fact_work_snapshot_test": "fact_work",
    "fact_shift_snapshot_test": "fact_shift",
    "fact_shrinkage_snapshot_test": "fact_shrinkage"
}

for source_table, target_table in promotions.items():

    source = (
        f"workforce_analytics.silver.{source_table}"
    )

    target = (
        f"workforce_analytics.silver.{target_table}"
    )

    (
        spark.table(source)
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target)
    )

    print(f"Promoted {source_table} to {target_table}")

In [0]:
fact_tables = [
    "fact_work",
    "fact_shift",
    "fact_shrinkage"
]

for table in fact_tables:

    print(f"\n========== {table} ==========")

    df = spark.table(
        f"workforce_analytics.silver.{table}"
    )

    display(
        df.groupBy("snapshot_date")
        .count()
        .orderBy("snapshot_date")
    )